In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import functools
from scipy.stats import spearmanr
print = functools.partial(print, flush=True)

DATA_FILES = [
    (1,  '../ha-1sec-full-rl-v4.pqt'),
    (2,  '../ha-2sec-full-rl-v4.pqt'),
    (3,  '../ha-3sec-full-rl-v4.pqt'),
    (4,  '../ha-4sec-full-rl-v4.pqt'),
    (8,  '../ha-8sec-full-rl-v4.pqt'),
    (16, '../ha-16sec-full-rl-v4.pqt'),
]
FRAME = 2
ORTHO_FILES = []
OUT_DIR = 'e5_out'
EPS = 1e-9
W_SCALE = 64
MIN_PER = 8
N_BLOCKS = 10
COVS = [0.05, 0.10, 0.25, 0.50, 1.00]
NUM_ROUNDS = 400
NUM_ROUNDS_METER = 300
PARAMS = {
    'objective': 'regression_l1',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_data_in_leaf': 500,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'force_row_wise': True,
    'verbose': -1,
    'seed': 7,
}

NORM_COLS = ['jmaD1', 'jmaD2', 'haBody', 'haWickTop', 'haWickBott', 'dBody_3']
FEATURES = ['jmaD1', 'haColour', 'jmaD2', 'rsxLast', 'rsxLastD1', 'rsxLastD2',
            'cfbD1', 'haBody', 'candleCross', 'wickAsym', 'haWickTop', 'haWickBott',
            'bodyRange', 'dBody_3', 'dWickTopR_3', 'dWickBotR_3', 'g']
LOAD_COLS = ['timestamp', 'date', 'Open', 'High', 'Low', 'jma', 'jmaD1', 'jmaD2', 'haColour',
             'rsxLast', 'rsxLastD1', 'rsxLastD2', 'cfbD1', 'haBody', 'haWickTop', 'haWickBott']

os.makedirs(OUT_DIR, exist_ok=True)

date_sets = []
for n, path in DATA_FILES:
    date_sets.append(set(pd.read_parquet(path, columns=['date'])['date'].unique()))
shared = np.array(sorted(set.intersection(*date_sets)))
starts = np.array([b[0] for b in np.array_split(shared, N_BLOCKS)])
print(f"shared days: {len(shared)}")


def prep(df):
    df['haRange'] = df['High'] - df['Low']
    df['candleCross'] = np.sign(df['Open'] - df['jma']).astype('int8')
    df['wickTop_body'] = df['haWickTop'] / (df['haBody'] + EPS)
    df['wickBot_body'] = df['haWickBott'] / (df['haBody'] + EPS)
    df['wickAsym'] = (df['haWickBott'] - df['haWickTop']) / (df['haBody'] + EPS)
    df['bodyRange'] = df['haBody'] / (df['haBody'] + df['haWickTop'] + df['haWickBott'] + EPS)
    gd = df.groupby(df['date'], sort=False)
    df['dBody_3'] = gd['haBody'].diff(3).fillna(0.0).values
    df['dWickTopR_3'] = gd['wickTop_body'].diff(3).fillna(0.0).values
    df['dWickBotR_3'] = gd['wickBot_body'].diff(3).fillna(0.0).values
    scale = gd['haRange'].transform(lambda x: x.rolling(W_SCALE, min_periods=MIN_PER).mean())
    df['scaleW'] = scale
    for c in NORM_COLS:
        df[c] = df[c] / scale


def segment(df):
    date = df['date'].to_numpy()
    s = pd.Series(np.sign(df['jmaD1'].to_numpy())).replace(0.0, np.nan)
    s = s.groupby(date).ffill().groupby(date).bfill().to_numpy()
    m = len(df)
    new_day = np.empty(m, dtype=bool); new_day[0] = True
    new_day[1:] = date[1:] != date[:-1]
    flip = np.empty(m, dtype=bool); flip[0] = True
    flip[1:] = s[1:] != s[:-1]
    st = np.flatnonzero(new_day | flip)
    lens = np.diff(np.append(st, m))
    g = np.arange(m) - np.repeat(st, lens) + 1
    df['g'] = g.astype(np.int64)
    df['segLen'] = np.repeat(lens, lens).astype(np.int32)
    df['remaining'] = (df['segLen'].to_numpy() - g).astype(np.float32)


path = dict(DATA_FILES)[FRAME]
df = pd.read_parquet(path, columns=LOAD_COLS)
df = df[df['date'].isin(shared)].reset_index(drop=True)
prep(df)
segment(df)
X = df[FEATURES].to_numpy(np.float32)
y = df['remaining'].to_numpy(np.float32)
g = df['g'].to_numpy(np.int64)
seglen = df['segLen'].to_numpy(np.int32)
dt = df['date'].to_numpy()
blk = (np.searchsorted(starts, dt, side='right') - 1).astype(np.int8)
ts = df['timestamp'].to_numpy()
sw = df['scaleW'].to_numpy(np.float32)
dayrel = (df['scaleW'] / df.groupby(df['date'], sort=False)['scaleW']
          .transform(lambda x: x.expanding().mean())).to_numpy(np.float32)
del df
n_rows = len(y)
print(f"frame {FRAME}s rows: {n_rows:,}")

ortho_mats = []
tier_names = ['price', '+vol']
bar_day = dt.astype('datetime64[D]')
for name, opath in ORTHO_FILES:
    o = pd.read_parquet(opath)
    o = o.sort_values('timestamp')
    ots = o['timestamp'].to_numpy()
    oday = ots.astype('datetime64[D]')
    cols = [c for c in o.columns if c != 'timestamp']
    vals = o[cols].to_numpy(np.float32)
    idx = np.searchsorted(ots, ts, side='right') - 1
    ok = (idx >= 0) & (oday[np.maximum(idx, 0)] == bar_day)
    Mo = vals[np.maximum(idx, 0)]
    Mo[~ok] = np.nan
    ortho_mats.append(Mo)
    tier_names.append(f'+{name}')
    print(f"ortho '{name}': {len(cols)} cols, matched {ok.mean() * 100:.1f}% of bars")

pred = np.full(n_rows, np.nan, np.float32)
q10 = np.full(n_rows, np.nan, np.float32)
q90 = np.full(n_rows, np.nan, np.float32)
abse = np.full(n_rows, np.nan, np.float32)
for f in range(1, N_BLOCKS):
    tr = blk < f
    te = blk == f
    mdl = lgb.train(PARAMS, lgb.Dataset(X[tr], label=y[tr]), num_boost_round=NUM_ROUNDS)
    p = mdl.predict(X[te]).astype(np.float32)
    pred[te] = p
    abse[te] = np.abs(p - y[te])
    for alpha, dst in ((0.1, q10), (0.9, q90)):
        qp = dict(PARAMS)
        qp['objective'] = 'quantile'
        qp['alpha'] = alpha
        dst[te] = lgb.train(qp, lgb.Dataset(X[tr], label=y[tr]),
                            num_boost_round=NUM_ROUNDS).predict(X[te]).astype(np.float32)
    print(f"primary fold {f} done  mae={abse[te].mean():.4f}")
width = q90 - q10

base_meta = np.hstack([X, pred[:, None], width[:, None]])
tier_mats = [base_meta, np.hstack([base_meta, sw[:, None], dayrel[:, None]])]
for Mo in ortho_mats:
    tier_mats.append(np.hstack([tier_mats[-1], Mo]))

methods = ['g_meter', 'width', 'pred_asc'] + [f'meter_{t}' for t in tier_names]
scores = {m: np.full(n_rows, np.nan, np.float32) for m in methods}
scores['width'] = width
scores['pred_asc'] = pred
for k in range(2, N_BLOCKS):
    trm = (blk >= 1) & (blk < k)
    tem = blk == k
    gt = g[trm]
    gmax = int(gt.max())
    cnt = np.bincount(gt, minlength=gmax + 1)
    sm = np.bincount(gt, weights=abse[trm], minlength=gmax + 1)
    lut = np.where(cnt > 0, sm / np.maximum(cnt, 1), abse[trm].mean())
    scores['g_meter'][tem] = lut[np.clip(g[tem], 0, gmax)].astype(np.float32)
    for tname, MX in zip(tier_names, tier_mats):
        mdl = lgb.train(PARAMS, lgb.Dataset(MX[trm], label=abse[trm]),
                        num_boost_round=NUM_ROUNDS_METER)
        scores[f'meter_{tname}'][tem] = mdl.predict(MX[tem]).astype(np.float32)
    print(f"meter block {k} done")

np.savez_compressed(os.path.join(OUT_DIR, 'scores.npz'), abse=abse, pred=pred,
                    width=width, g=g, seglen=seglen, blk=blk,
                    **{f's_{m}': scores[m] for m in methods})

eval_blocks = list(range(2, N_BLOCKS))
rows = []
for m in methods:
    s = scores[m]
    for k in eval_blocks:
        idx = np.flatnonzero(blk == k)
        order = idx[np.argsort(s[idx], kind='stable')]
        for c in COVS:
            sel = order[:max(1, int(c * len(order)))]
            rows.append((m, k, c, float(abse[sel].mean())))
res = pd.DataFrame(rows, columns=['method', 'block', 'cov', 'mae'])
tab = res.pivot_table(index='method', columns='cov', values='mae', aggfunc='mean')
tab = tab.loc[methods]
print("\nselective MAE (mean over eval blocks 2-8):")
print(tab.round(4))
tab.to_csv(os.path.join(OUT_DIR, 'selective_mae.csv'))

emask = blk >= 2
print("\nspearman(score, abs_e), pooled eval blocks:")
for m in methods:
    rho = spearmanr(scores[m][emask], abse[emask]).statistic
    print(f"  {m:>14}: {rho:.4f}")

print("\ncomposition of selected 25% (pooled eval blocks):")
comp = [('all', np.flatnonzero(emask))]
for m in methods:
    idx = np.flatnonzero(emask)
    order = idx[np.argsort(scores[m][idx], kind='stable')]
    comp.append((m, order[:int(0.25 * len(order))]))
crows = []
for name, sel in comp:
    crows.append((name, float(g[sel].mean()), float(y[sel].mean()),
                  float(seglen[sel].mean()), float(abse[sel].mean())))
cdf = pd.DataFrame(crows, columns=['method', 'mean_g', 'mean_remaining', 'mean_segLen', 'mae'])
print(cdf.round(3).to_string(index=False))
cdf.to_csv(os.path.join(OUT_DIR, 'composition_25.csv'), index=False)

fig, ax = plt.subplots(figsize=(7, 5))
for m in methods:
    t = tab.loc[m]
    ax.plot(t.index, t.values, 'o-', label=m)
ax.set_xscale('log')
ax.set_xlabel('coverage')
ax.set_ylabel('selective MAE (bars)')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'selective_mae.png'), dpi=130)
plt.show()

shared days: 1155
frame 2s rows: 14,247,372
